In [5]:
import openai
import json

def generate_training_data(query, api_key, num_samples=5):
    openai.api_key = api_key
    
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Generate a training example based on the given query. These examples represent the kind of geospatial questions that a user might ask."},
            {"role": "user", "content": query}
        ]
    )
    return response.choices[0].message.content

In [6]:
KEY = "***REDACTED***"

EXAMPLES = '''Please generate the training data for a 13-way classification task following these instructions:
Rule 1: Please return the result in a JSON format.
Rule 2: There are 13 action categories. Examples questions for each category are provided below.
{
    "action": "retrieve", #directly retrieve value from the data
    "query": "What’s the population density of {state1}?"
},
{
    "action": "compare", #compare values of multiple data
    "query": "Which state has higher population density, {state2} or {state3}?"
},
{
    "action": "find extremum", # Finding min/max values
    "query": "Which state has the {highest/lowest} population density?"
},
{
    "action": "aggregated functions",  # Calculating averages and related metrics
    "queries": "What's the average population density in this map?" # or "What’s the state closest to the average population density?"
},
{
    "action": "filter", # Filtering based on conditions
    "query": "Which state has a population density of lower than 100 people per square mile?"
},
{
    "action": "sort", # Ordering and ranking
    "query": "What’s the top 3 states with the highest population density?"
},
{
    "action": "data ranges",  # Finding data ranges
    "query": "What is the range of population density in the United States?"
},
{
    "action": "characterize data distribution",  # Distribution analysis
    "query": "TBD, (e.g. Across the world, are there more countries with a low GDP, medium GDP, or a high GDP?)"
},
{
    "action": "cluster",  # Finding similar values/groups
    "query": "Which state has a similar population density to New York?"
},
{
    "action": "is pattern", # Pattern recognition
    "queries": "Is there a pattern in this map? (run Moran’s I)"
},

{
    "action": "describe pattern", # Pattern description
    "queries": "Can you describe the pattern in this map? (get LISA clusters)"
},
{
    "action": "find outliers", # Finding outliers
    "query": "What states have high population density despite being surrounded by low population density?"
},
{
    "action": "correlate", # Relationship analysis
    "query": "Is there a relationship between income and population density?"
},
{
    "action": "others",  # No-data situations
    "queries": "What’s population density?" # or "What’s the population density of a banana?", or "What’s a choropleth map?"
}
Rule 3: I only have these query examples. Your goal is to use these examples to generate a diverse set of training data for me to train a classifer that has not been captured in the examples above. 
Rule 4: Please generate 1 samples for each category.
'''

generate_training_data(EXAMPLES, KEY, num_samples=1)

'```json\n[\n    {\n        "action": "Retrieve values",\n        "query": "What’s the population density of California?"\n    },\n    {\n        "action": "Retrieve and compare",\n        "query": "Which state has higher population density, Texas or Florida?"\n    },\n    {\n        "action": "Find extremum",\n        "query": "Which state has the lowest population density?"\n    },\n    {\n        "action": "Compute derived values",\n        "query": "What’s the state closest to the highest population density?"\n    },\n    {\n        "action": "Filter",\n        "query": "Which states have a population density greater than 300 people per square mile?"\n    },\n    {\n        "action": "Sort",\n        "query": "What are the bottom 5 states with the lowest population density?"\n    },\n    {\n        "action": "Determine data ranges",\n        "query": "What is the population density range for the Northeastern United States?"\n    },\n    {\n        "action": "Characterize data distr

In [9]:
from transformers import pipeline

classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Define your 13 classes
candidate_labels = [
    "retrieve", #directly retrieve value from the data
    "compare",           # Direct comparison between states
    "find extremum",           # Finding min/max values
    "aggregated functions",         # Calculating averages and related metrics
    "filter",            # Filtering based on conditions
    "sort",              # Ordering and ranking
    "data ranges",        # Finding data ranges
    "characterize data distribution",   # Distribution analysis
    "cluster",          # Finding similar values/groups
    "identify the geospatial pattern",      # Pattern recognition
    "describe the geospatial pattern",       # Pattern description
    "find outliers",           # Finding outliers
    "correlate",    # Relationship analysis
    "others"     # No-data situations
]

# Classify a text
text = "What’s population density? "
result = classifier(text, candidate_labels)
print(result['labels'][0])  # Most likely class
print(result['scores'][0]) 

Device set to use cpu


compare values of multiple data
0.4212968945503235


In [56]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import logging
import os
import re
import spacy
from typing import List, Dict, Tuple, Any

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Load spaCy model
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    logger.info("Downloading spaCy model...")
    os.system('python -m spacy download en_core_web_sm')
    nlp = spacy.load('en_core_web_sm')

def set_seed(seed: int = 42) -> None:
    """Set seeds for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)

def replace_named_entities(text: str) -> str:
    """
    Replace named entities in text with generic tokens.
    
    Args:
        text: Input text string
        
    Returns:
        Text with named entities replaced
    """
    doc = nlp(text)
    words = []
    for token in doc:
        # Replace geographical entities with LOCATION token
        if token.ent_type_ in ['GPE', 'LOC']:
            words.append('LOCATION')
        else:
            words.append(token.text)
    return ' '.join(words)

def clean_text(text: str) -> str:
    """
    Clean and preprocess text data with named entity replacement.
    
    Args:
        text: Input text string
        
    Returns:
        Cleaned text string
    """
    # Convert to lowercase
    text = text.lower()
    
    # Replace named entities
    text = replace_named_entities(text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    # Remove special characters but keep necessary punctuation
    text = re.sub(r'[^a-z0-9\s?.!,]', '', text)
    
    # Standardize question marks
    text = re.sub(r'\?+', '?', text)
    
    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def clean_label(label: str) -> str:
    """
    Clean and standardize label strings.
    
    Args:
        label: Input label string
        
    Returns:
        Cleaned label string
    """
    # Convert to lowercase for consistency
    label = label.lower()
    
    # Remove any special characters or spaces
    label = re.sub(r'[^a-z0-9_]', '_', label)
    
    # Remove multiple underscores
    label = re.sub(r'_+', '_', label)
    
    return label.strip('_')

class SpatialQueriesDataset(Dataset):
    """Custom dataset for spatial queries classification."""
    
    def __init__(self, queries: List[str], labels: List[int], tokenizer: Any, max_length: int = 128):
        if len(queries) != len(labels):
            raise ValueError("Number of queries and labels must match")
        if not all(isinstance(q, str) for q in queries):
            raise ValueError("All queries must be strings")
            
        # Clean queries and replace named entities before tokenization
        cleaned_queries = [clean_text(q) for q in queries]
        
        # Log some examples of cleaned queries for verification
        if len(cleaned_queries) > 0:
            sample_size = min(5, len(cleaned_queries))
            logger.info("Sample of cleaned queries:")
            for i in range(sample_size):
                logger.info(f"Original: {queries[i]}")
                logger.info(f"Cleaned : {cleaned_queries[i]}\n")
            
        self.encodings = tokenizer(
            cleaned_queries,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = labels

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self) -> int:
        return len(self.labels)

def validate_data(data: List[Dict[str, str]]) -> None:
    """
    Validate input data format and contents.
    
    Args:
        data: List of dictionaries containing 'query' and 'label' keys
        
    Raises:
        ValueError: If data format is invalid
    """
    if not isinstance(data, list):
        raise ValueError("Data must be a list of dictionaries")
    
    for item in data:
        if not isinstance(item, dict):
            raise ValueError("Each item in data must be a dictionary")
        if 'query' not in item or 'action' not in item:
            raise ValueError("Each dictionary must contain 'query' and 'action' keys")
        if not isinstance(item['query'], str) or not isinstance(item['action'], str):
            raise ValueError("Query and label must be strings")
        if not item['query'].strip():
            raise ValueError("Query cannot be empty")
        if not item['action'].strip():
            raise ValueError("Label cannot be empty")

def prepare_data(data: List[Dict[str, str]]) -> Tuple[List[str], List[int], Dict[str, int], Dict[int, str]]:
    """
    Prepare and clean the data for training.
    
    Args:
        data: List of dictionaries containing 'query' and 'label' keys
        
    Returns:
        Tuple containing queries, labels, label2id mapping, and id2label mapping
    """
    # Validate input data
    validate_data(data)
    
    if not data:
        raise ValueError("Empty data provided")
    
    # Clean and standardize labels
    cleaned_labels = [clean_label(item['action']) for item in data]
    
    unique_labels = sorted(list(set(cleaned_labels)))
    if len(unique_labels) < 2:
        raise ValueError("Need at least 2 unique labels for classification")
    
    label2id = {label: i for i, label in enumerate(unique_labels)}
    id2label = {i: label for label, i in label2id.items()}
    
    # Clean queries and convert labels to ids
    queries = [item['query'] for item in data]
    labels = [label2id[clean_label(item['action'])] for item in data]
    
    logger.info(f"Prepared {len(queries)} queries with {len(unique_labels)} unique labels")
    logger.info(f"Labels: {unique_labels}")
    
    # Log some examples of queries and their labels
    sample_size = min(5, len(queries))
    logger.info("\nSample of query-label pairs:")
    for i in range(sample_size):
        logger.info(f"Query: {queries[i]}")
        logger.info(f"Label: {id2label[labels[i]]}\n")
    
    return queries, labels, label2id, id2label

def compute_metrics(pred) -> Dict[str, float]:
    """Compute evaluation metrics."""
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted'),
        'precision': precision_score(labels, preds, average='weighted'),
        'recall': recall_score(labels, preds, average='weighted')
    }

def validate_model_fairness(model: Any, tokenizer: Any, id2label: Dict[int, str]) -> Tuple[bool, Dict[str, str]]:
    """
    Validate model's fairness across different locations.
    
    Args:
        model: Trained model
        tokenizer: Tokenizer
        id2label: Mapping from label ids to label strings
        
    Returns:
        Tuple of (is_fair: bool, results: Dict[str, str])
    """
    test_queries = [
        "Are old people clustered in Idaho?",
        "Are old people clustered in California?",
        "Are elderly people clustered in Oregon?",
        "Are senior citizens clustered in Washington?",
        "Is there a concentration of seniors in Nevada?",
        "Are retirees clustered in Arizona?"
    ]
    
    results = {}
    for query in test_queries:
        # Clean and tokenize
        cleaned_query = clean_text(query)
        inputs = tokenizer(cleaned_query, return_tensors="pt", truncation=True, padding=True)
        
        # Get prediction
        outputs = model(**inputs)
        predicted_label = id2label[outputs.logits.argmax().item()]
        
        results[query] = predicted_label
        logger.info(f"\nTest query: {query}")
        logger.info(f"Cleaned query: {cleaned_query}")
        logger.info(f"Predicted label: {predicted_label}")
    
    # Check for consistency
    labels = list(results.values())
    is_consistent = all(label == labels[0] for label in labels)
    
    if not is_consistent:
        logger.warning("\nInconsistent predictions across locations detected!")
    else:
        logger.info("\nModel predictions are consistent across different locations.")
    
    return is_consistent, results

def train_model(
    data: List[Dict[str, str]],
    output_dir: str = "./spatial_classifier",
    model_name: str = "distilroberta-base",
    num_epochs: int = 5,
    batch_size: int = 8,
    learning_rate: float = 1e-4,
) -> Tuple[Any, Any, Dict[str, int], Dict[int, str]]:
    
    # Set seed for reproducibility
    set_seed()
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize tokenizer
    logger.info(f"Loading tokenizer: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
    
    # Prepare data
    queries, labels, label2id, id2label = prepare_data(data)
    
    # Split data
    train_queries, val_queries, train_labels, val_labels = train_test_split(
        queries, labels, test_size=0.2, random_state=42, stratify=labels
    )
    
    # Create datasets
    train_dataset = SpatialQueriesDataset(train_queries, train_labels, tokenizer)
    val_dataset = SpatialQueriesDataset(val_queries, val_labels, tokenizer)
    
    # Initialize model
    logger.info(f"Loading model: {model_name}")
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(label2id),
        label2id=label2id,
        id2label=id2label
    )
    
    # Define training arguments
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        warmup_steps=100,
        weight_decay=0.01,
        logging_dir=os.path.join(output_dir, 'logs'),
        logging_steps=50,
        evaluation_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        learning_rate=learning_rate,
        gradient_accumulation_steps=4,
        max_grad_norm=1.0,
        dataloader_num_workers=0,
        fp16=False,
        report_to="none"
    )
    
    # Define trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )
    
    # Train the model
    logger.info("Starting training...")
    trainer.train()
    
    # Validate model fairness
    logger.info("\nValidating model fairness across different locations...")
    is_fair, fairness_results = validate_model_fairness(model, tokenizer, id2label)
    
    # Save the model and tokenizer
    logger.info(f"\nSaving model and tokenizer to {output_dir}")
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    
    return model, tokenizer, label2id, id2label

def predict(model: Any, tokenizer: Any, query: str, id2label: Dict[int, str]) -> str:
    """
    Make a prediction for a single query.
    
    Args:
        model: Trained model
        tokenizer: Tokenizer
        query: Input query string
        id2label: Mapping from label ids to label strings
        
    Returns:
        Predicted label string
    """
    # Clean the query and replace named entities
    cleaned_query = clean_text(query)
    print(cleaned_query)
    
    # Log the cleaning process
    logger.info(f"Original query: {query}")
    logger.info(f"Cleaned query: {cleaned_query}")
    
    # Tokenize
    inputs = tokenizer(cleaned_query, return_tensors="pt", truncation=True, padding=True)
    
    # Get prediction
    outputs = model(**inputs)
    predicted_id = outputs.logits.argmax().item()
    predicted_label = id2label[predicted_id]
    
    logger.info(f"Predicted label: {predicted_label}")
    
    return predicted_label

In [64]:
# Load your data
# with open("dataset.json", "r") as f:
#     data = json.load(f)

# # Train model
# model, tokenizer, label2id, id2label = train_model(data)

# Test the model with different locations
test_queries = [
    "Are old people clustered in Idaho?",
    "Are old people clustered in California?",
    "Are elderly people clustered in Oregon?"
]

for query in test_queries:
    result = predict(model, tokenizer, query, id2label)
    print(f"\nQuery: {query}")
    print(f"Prediction: {result}")

INFO:__main__:Original query: Are old people clustered in Idaho?
INFO:__main__:Cleaned query: are old people clustered in idaho ?
INFO:__main__:Predicted label: correlate
INFO:__main__:Original query: Are old people clustered in California?
INFO:__main__:Cleaned query: are old people clustered in ?
INFO:__main__:Predicted label: is_pattern
INFO:__main__:Original query: Are elderly people clustered in Oregon?
INFO:__main__:Cleaned query: are elderly people clustered in ?
INFO:__main__:Predicted label: compare


are old people clustered in idaho ?

Query: Are old people clustered in Idaho?
Prediction: correlate
are old people clustered in ?

Query: Are old people clustered in California?
Prediction: is_pattern
are elderly people clustered in ?

Query: Are elderly people clustered in Oregon?
Prediction: compare


In [50]:
test_query = "Are old people clustered in Idaho?"
inputs = tokenizer(test_query.lower(), return_tensors="pt", truncation=True, padding=True)
outputs = model(**inputs)
predicted_label = id2label[outputs.logits.argmax().item()]
logger.info(f"Test query: {test_query}")
logger.info(f"Predicted label: {predicted_label}")

INFO:__main__:Test query: Are old people clustered in Idaho?
INFO:__main__:Predicted label: correlate


In [ ]:
# What's the state closest to the average population density?
# Is there a pattern in Washington?
# Is there a spatial autocorrelation in Washington?
# can you describe the pattern in this map?